# Поиск автоматизированного трафика по событиям за сутки

**Задача.** Для каждой `cookie_id` из `test.csv` сформировать `score` от 0 до 1.
Больший score должен соответствовать большему риску автоматизированного сбора
данных. Целевая метка есть только в `train.csv`.

**Идея.** Сначала понять, какие события доступны на момент окончания окна.
После этого собрать по одной строке признаков на куку. Проверить простую
модель с двумя признаками, затем расширить признаки и проверить две модели
на днях позже обучающих. В конце обучить их на всём train и усреднить оценки.

Вычисления выполняются локально. В модели не используются LLM, внешние API,
ручная разметка или исходный идентификатор куки как признак. Использованы
Python 3.12.14, NumPy 2.3.5, pandas 2.2.3, scikit-learn 1.8.0 и
предоставленный `metric.py`. `random_state=42`.

( Положить `train.csv`, `test.csv`, `events.csv` в папку `data` рядом с ноутбуком. Положить рядом официальный
`metric.py`.)


## 1. Импорт и загрузка

Таблицы train и test содержат границы окна наблюдения; события лежат отдельно.
Соединяем метаданные только ради построения признаков. Метки test отсутствуют.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import warnings
from pandas.errors import PerformanceWarning
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from metric import precision_at_recall

# Многократное добавление признаков вызывает лишь предупреждение о скорости pandas.
warnings.filterwarnings("ignore", category=PerformanceWarning)

RANDOM_STATE = 42
EVENT_TYPES = ["item_view", "search_results_view", "photo_swipe", "favorite_add",
               "seller_page_view", "contact_phone_show", "captcha_shown", "login",
               "contact_chat_open", "contact_message_sent"]
DATA_DIR = Path("data")
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
events_path = DATA_DIR / "events.csv"
if not events_path.exists():
    events_path = DATA_DIR / "events.csv.gz"
events = pd.read_csv(events_path, parse_dates=["event_ts"])
meta = pd.concat([train.drop(columns="target"), test], ignore_index=True)
assert meta.cookie_id.is_unique
print("train / test / events:", train.shape, test.shape, events.shape)
print("Доля положительного класса:", round(train.target.mean(), 4))
print("Периоды train / test:", train.window_start_ts.min(), train.window_start_ts.max(),
      "/", test.window_start_ts.min(), test.window_start_ts.max())


train / test / events: (11091, 5) (4909, 4) (328905, 14)
Доля положительного класса: 0.0811
Периоды train / test: 2026-04-06 2026-04-19 / 2026-04-20 2026-04-26


## 2. Проверка данных перед обучением

Смотрим на пропуски и технические особенности. Поля объявления или поисковой
выдачи пустуют у событий другого типа — это ожидаемо, поэтому удалять все
строки с пропусками нельзя. Встречаются разные написания платформы (`web`,
`Web`, `WEB`), а также одинаковые во всех полях строки событий. Их убираем,
чтобы не учитывать одну и ту же запись дважды. Различающиеся события, включая
действия в одну секунду, сохраняем.

Отдельно считаем события вне окна. Их наличие — причина фильтровать время
для **каждой** куки, а не только по общей дате train/test.


In [2]:
print("Пропуски по колонкам событий:")
display(events.isna().mean().sort_values(ascending=False).round(3).to_frame("доля"))
print("Варианты platform:", sorted(events.platform.dropna().unique()))
print("Точных дубликатов строк:", int(events.duplicated().sum()))

window_check = events[["cookie_id", "event_ts"]].merge(
    meta[["cookie_id", "window_start_ts", "window_end_ts"]], on="cookie_id")
inside = ((window_check.event_ts >= pd.to_datetime(window_check.window_start_ts)) &
          (window_check.event_ts < pd.to_datetime(window_check.window_end_ts)))
print("Событий вне окна:", int((~inside).sum()))
del window_check, inside

events = events.drop_duplicates()
print("После удаления точных дубликатов:", len(events), "событий")


Пропуски по колонкам событий:


,доля
search_query,0.695
search_page,0.695
pointer_x,0.670
pointer_y,0.670
seller_type,0.407
item_id,0.348
item_category,0.100
item_location,0.072
user_agent,0.000
platform,0.000


Варианты platform: ['ANDROID', 'Android', 'IOS', 'WEB', 'Web', 'android', 'desktop', 'iOS', 'ios', 'iphone', 'web']
Точных дубликатов строк: 4863
Событий вне окна: 40779
После удаления точных дубликатов: 324042 событий


## 3. Агрегирование событий в признаки

**Главная защита от утечки времени:** берём только события
`window_start_ts <= event_ts < window_end_ts` для соответствующей куки.
Никакие будущие события при её оценке не используются.

Каждая строка итоговой таблицы — одна кука. Признаки отражают:

- объём и состав действий: просмотры выдачи/объявлений, контакты, капча и т. д.;
- разнообразие и повторяемость объявлений, категорий, запросов, географии;
- скорость, регулярность и всплески событий, их переходы друг в друга;
- обобщённые характеристики платформы и User-Agent без сырой строки.

Сырые строки `item_category`, `item_location`, `search_query`, `user_agent`
могут обозначать конкретное объявление или клиента и меняться в будущем.
Поэтому берём их количество, повторяемость, длину запроса и общие свойства
User-Agent, а не подаём произвольные идентификаторы и строки напрямую.
Платформу приводим к нижнему регистру, `iphone` объединяем с `ios`.

Часть агрегатов невозможно посчитать у куки без нужных событий. Такие значения
заменяем на −1. Частоты событий, которые не встретились, остаются нулевыми.
`cookie_id` служит только ключом для соединения и не входит в матрицу модели.
Одна функция применяется ко всем кукам, без использования test-меток.


In [3]:
def describe_features(meta: pd.DataFrame, events: pd.DataFrame) -> pd.DataFrame:
    """Собрать одну строку числовых признаков на куку из её суточного окна."""
    meta = meta.copy()
    for col in ["cookie_created_at", "window_start_ts", "window_end_ts"]:
        meta[col] = pd.to_datetime(meta[col])
    # Сначала присоединяем границы окна: у куки могут быть события и вне этого дня.
    ev = events.merge(meta[["cookie_id", "window_start_ts", "window_end_ts"]],
                      on="cookie_id", how="inner", validate="many_to_one")
    ev = ev.loc[(ev.event_ts >= ev.window_start_ts) &
                (ev.event_ts < ev.window_end_ts)].copy()
    # Упорядочиваем события куки по времени, сохраняя порядок при равном времени.
    ev = ev.sort_values(["cookie_id", "event_ts"], kind="stable")
    ev["event_name"] = ev.event_name.fillna("unknown")
    ev["platform_clean"] = ev.platform.str.lower().replace({"iphone": "ios"})
    ev["hour"] = ev.event_ts.dt.hour
    ev["minute_of_day"] = ev.hour * 60 + ev.event_ts.dt.minute
    ev["second_of_day"] = ev.minute_of_day * 60 + ev.event_ts.dt.second
    ev["offset_s"] = (ev.event_ts - ev.window_start_ts).dt.total_seconds()
    ev["delta_s"] = ev.groupby("cookie_id").event_ts.diff().dt.total_seconds()
    ev["item_repeat"] = ev.duplicated(["cookie_id", "item_id"]) & ev.item_id.notna()
    ev["query_length"] = ev.search_query.fillna("").str.len()
    ev["query_words"] = ev.search_query.fillna("").str.split().str.len()
    ev["ua_length"] = ev.user_agent.fillna("").str.len()
    ev["page_num"] = pd.to_numeric(ev.search_page, errors="coerce")
    ev["pointer_x"] = pd.to_numeric(ev.pointer_x, errors="coerce")
    ev["pointer_y"] = pd.to_numeric(ev.pointer_y, errors="coerce")

    # Все агрегаты считаются отдельно для каждой куки, без использования target.
    g = ev.groupby("cookie_id", sort=False)
    f = pd.DataFrame(index=meta.cookie_id)
    f.index.name = "cookie_id"
    f["cookie_age_days"] = ((meta.window_start_ts - meta.cookie_created_at)
                             .dt.total_seconds() / 86400).to_numpy()
    f["cookie_created_hour"] = meta.cookie_created_at.dt.hour.to_numpy()
    f["window_weekday"] = meta.window_start_ts.dt.dayofweek.to_numpy()
    f["n_events"] = g.size()
    f["active_span_s"] = (g.event_ts.max() - g.event_ts.min()).dt.total_seconds()

    # Разнообразие действий, доля самого частого значения и наличие полей.
    for c in ["event_name", "eid", "platform_clean", "user_agent", "item_id",
              "item_category", "item_location", "seller_type", "search_query",
              "search_page", "hour", "minute_of_day"]:
        f[c + "_unique"] = g[c].nunique()
    for c in ["event_name", "platform_clean", "item_category", "item_location",
              "seller_type", "search_query", "item_id"]:
        vc = ev.groupby(["cookie_id", c], observed=True).size()
        f[c + "_top_share"] = vc.groupby(level=0).max() / f.n_events
    for c in ["item_id", "search_query", "pointer_x", "item_location"]:
        f[c + "_present_rate"] = g[c].count() / f.n_events
    # Количество и доля каждого действия различают просмотры, контакты и капчу.
    for c in EVENT_TYPES:
        f["n_" + c] = ev.event_name.eq(c).groupby(ev.cookie_id).sum()
        f["rate_" + c] = f["n_" + c] / f.n_events
    for c in ["desktop", "web", "android", "ios"]:
        f["n_platform_" + c] = ev.platform_clean.eq(c).groupby(ev.cookie_id).sum()
        f["rate_platform_" + c] = f["n_platform_" + c] / f.n_events
    f["item_views_per_unique"] = f.n_item_view / (f.item_id_unique + 1)
    f["searches_per_query"] = f.n_search_results_view / (f.search_query_unique + 1)
    f["search_per_item_view"] = f.n_search_results_view / (f.n_item_view + 1)
    f["contact_per_item_view"] = (f.n_contact_phone_show + f.n_contact_chat_open) / (f.n_item_view + 1)
    f["photo_per_item_view"] = f.n_photo_swipe / (f.n_item_view + 1)
    f["events_per_active_minute"] = f.n_events / (f.active_span_s / 60 + 1)
    f["item_repeat_rate"] = g.item_repeat.mean()
    # Время, номера страниц и длина запросов помогают различать куки
    # даже при одинаковом общем числе событий.
    for name, vals in {"search_page": "page_num", "query_length": "query_length",
                       "query_words": "query_words", "pointer_x": "pointer_x",
                       "pointer_y": "pointer_y", "offset_s": "offset_s",
                       "hour": "hour", "ua_length": "ua_length"}.items():
        for stat in ["min", "max", "mean", "std"]:
            f[name + "_" + stat] = getattr(g[vals], stat)()
    for boundary, hours in [("night", [0, 1, 2, 3, 4, 5]),
                            ("morning", [6, 7, 8, 9, 10, 11]),
                            ("evening", [18, 19, 20, 21, 22, 23])]:
        f["rate_" + boundary] = ev.hour.isin(hours).groupby(ev.cookie_id).mean()
    for threshold in [0, 1, 2, 5, 10, 30, 60, 300, 1800]:
        f[f"delta_le_{threshold}_rate"] = ev.delta_s.le(threshold).groupby(ev.cookie_id).sum() / (f.n_events - 1).clip(lower=1)
    dg = g.delta_s
    for stat in ["mean", "std", "median", "min", "max"]:
        f["delta_" + stat] = getattr(dg, stat)()
    for q in [.1, .25, .75, .9]:
        f[f"delta_q{int(100*q)}"] = dg.quantile(q)
    f["delta_cv"] = f.delta_std / (f.delta_mean + 1)
    f["delta_q90_q10_ratio"] = f.delta_q90 / (f.delta_q10 + 1)
    f["active_minutes"] = g.minute_of_day.nunique()
    f["events_per_active_minute_bucket"] = f.n_events / f.active_minutes

    # Максимум событий за минуту и за пять минут характеризует всплески активности.
    ev["minute_bucket"] = ev.event_ts.dt.floor("min")
    ev["five_minute_bucket"] = ev.event_ts.dt.floor("5min")
    for bucket in ["minute_bucket", "five_minute_bucket"]:
        counts = ev.groupby(["cookie_id", bucket]).size()
        by_cookie = counts.groupby(level=0)
        f[bucket + "_max_events"] = by_cookie.max()
        f[bucket + "_std_events"] = by_cookie.std()
        f[bucket + "_many_events"] = counts.ge(5).groupby(level=0).sum()

    # Считаем характерные переходы между соседними по времени действиями.
    prev_type = g.event_name.shift()
    for a, b in [("search_results_view", "item_view"),
                 ("item_view", "item_view"),
                 ("search_results_view", "search_results_view"),
                 ("item_view", "photo_swipe"),
                 ("item_view", "contact_phone_show"),
                 ("item_view", "seller_page_view")]:
        f[f"transition_{a}_to_{b}"] = ((prev_type == a) & (ev.event_name == b)).groupby(ev.cookie_id).sum()
    f["same_item_consecutive_rate"] = ((g.item_id.shift() == ev.item_id) & ev.item_id.notna()).groupby(ev.cookie_id).sum() / f.n_events
    f["same_query_consecutive_rate"] = ((g.search_query.shift() == ev.search_query) & ev.search_query.notna()).groupby(ev.cookie_id).sum() / f.n_events
    f["sequential_page_rate"] = ((ev.page_num - g.page_num.shift() == 1) &
                                  (g.search_query.shift() == ev.search_query) &
                                  ev.search_query.notna()).groupby(ev.cookie_id).sum() / f.n_events
    for c in ["item_view", "search_results_view", "contact_phone_show"]:
        subset = ev.loc[ev.event_name.eq(c)]
        if not subset.empty:
            sg = subset.groupby("cookie_id")
            f[c + "_span_s"] = (sg.event_ts.max() - sg.event_ts.min()).dt.total_seconds()
            f[c + "_delta_median"] = sg.event_ts.diff().dt.total_seconds().groupby(subset.cookie_id).median()
            f[c + "_category_unique"] = sg.item_category.nunique()
            f[c + "_location_unique"] = sg.item_location.nunique()

    # Из первого User-Agent берём общие признаки клиента, не кодируя исходную строку.
    ua = g.user_agent.first().reindex(f.index).fillna("")
    ua_l = ua.str.lower()
    for name, pattern in {"headless": "headless|bot|spider|crawler|python|curl|wget",
                          "mobile": "mobile|android|iphone|ipad",
                          "windows": "windows", "mac": "macintosh|mac os",
                          "linux": "linux", "chrome": "chrome/|crios/",
                          "firefox": "firefox/", "safari": "safari/",
                          "app": "avito|okhttp|dalvik"}.items():
        f["ua_" + name] = ua_l.str.contains(pattern, regex=True).astype(int).to_numpy()
    browser_version = ua.str.extract(r"(?:Chrome|Firefox|Version|CriOS)/([0-9]+)", expand=False)
    f["ua_browser_major"] = pd.to_numeric(browser_version, errors="coerce").to_numpy()
    f = f.replace([np.inf, -np.inf], np.nan)
    return f.fillna(-1).astype("float32")


In [4]:
x = describe_features(meta, events)
X_train = x.loc[train.cookie_id]
X_test = x.loc[test.cookie_id]
y = train.target.to_numpy()
print("Число куки и признаков:", x.shape)
display(X_train.head(3).iloc[:, :20])


Число куки и признаков: (16000, 157)


,cookie_age_days,cookie_created_hour,window_weekday,n_events,active_span_s,event_name_unique,eid_unique,platform_clean_unique,user_agent_unique,item_id_unique,item_category_unique,item_location_unique,seller_type_unique,search_query_unique,search_page_unique,hour_unique,minute_of_day_unique,event_name_top_share,platform_clean_top_share,item_category_top_share
cookie_id,,,,,,,,,,,,,,,,,,,,
ck_54a059eb7d3ea68b,135.603699,9.0,0.0,7.0,167.0,3.0,3.0,1.0,1.0,4.0,3.0,5.0,2.0,3.0,3.0,1.0,4.0,0.428571,1.000000,0.571429
ck_7e4de46eeab82974,194.576111,10.0,0.0,41.0,39476.0,7.0,7.0,2.0,1.0,19.0,1.0,12.0,2.0,5.0,6.0,8.0,30.0,0.536585,0.756098,0.951219
ck_9320229ef6304522,32.994423,0.0,0.0,36.0,36491.0,9.0,9.0,2.0,1.0,19.0,7.0,9.0,2.0,10.0,7.0,7.0,21.0,0.361111,0.777778,0.222222


## 4. Проверка по времени и официальной метрике

Test датирован позже train, поэтому мы имитируем этот сценарий. Сначала учим
на днях до 14 апреля и проверяем 14–16 апреля, затем учим на днях до 17 апреля
и проверяем 17–19 апреля. Таким образом, в проверке модель никогда не видит
метки будущего периода при обучении.

Основная метрика — **максимальная precision при recall не ниже 0,7**. Порог
выбирается проверяющей системой; для самопроверки вызывается её официальный
`precision_at_recall`, который правильно учитывает одинаковые score. PR-AUC и
ROC-AUC показаны только для диагностики.

Положительный класс занимает около 8% train. Поэтому кроме precision важно
контролировать recall: если отмечать лишь несколько очевидных ботов, легко
получить высокую precision, но не выполнить условие 70% найденных ботов.


In [5]:
def models():
    return {
        "hist": HistGradientBoostingClassifier(max_iter=360, learning_rate=.045,
            max_leaf_nodes=15, l2_regularization=8, min_samples_leaf=25,
            random_state=RANDOM_STATE),
        "trees": ExtraTreesClassifier(n_estimators=500, min_samples_leaf=3,
            max_features=.85, n_jobs=-1, random_state=RANDOM_STATE),
    }

def evaluate(y, pred):
    return (round(precision_at_recall(y, pred), 4),
            round(average_precision_score(y, pred), 4),
            round(roc_auc_score(y, pred), 4))


### Простая отправная точка

Сначала используем только общее число событий и количество уникальных
объявлений, как в quickstart. Это показывает, достаточно ли одной интенсивности
активности. Модель — случайный лес с теми же временными разбиениями. Метрика
считается по кукам, а не событиям.


In [6]:
dates = pd.to_datetime(train.window_start_ts).to_numpy()
baseline_columns = ["n_events", "item_id_unique"]
for start, end in [("2026-04-14", "2026-04-17"),
                   ("2026-04-17", "2026-04-20")]:
    fit = dates < np.datetime64(start)
    valid = (dates >= np.datetime64(start)) & (dates < np.datetime64(end))
    baseline = RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                      random_state=RANDOM_STATE, n_jobs=-1)
    baseline.fit(X_train.iloc[fit][baseline_columns], y[fit])
    base_score = baseline.predict_proba(X_train.iloc[valid][baseline_columns])[:, 1]
    print(start, "baseline (P@R>=0.7, PR-AUC, ROC-AUC):",
          evaluate(y[valid], base_score))


2026-04-14 baseline (P@R>=0.7, PR-AUC, ROC-AUC): (0.1032, np.float64(0.2258), np.float64(0.6205))
2026-04-17 baseline (P@R>=0.7, PR-AUC, ROC-AUC): (0.0931, np.float64(0.1627), np.float64(0.5749))


Baseline даёт P@R≥0,7 около **0,10** на обоих периодах. Один счётчик событий
не показывает, как именно кука двигалась по объявлениям и с какой скоростью.
Теперь проверим признаки последовательностей и интервалов в более богатых
моделях. HistGradientBoosting улавливает нелинейные сочетания признаков, а
ExtraTrees строит другой набор разбиений. Сравним их и усреднение, не подбирая
порог на test.


In [7]:
for start, end in [("2026-04-14", "2026-04-17"),
                   ("2026-04-17", "2026-04-20")]:
    fit = dates < np.datetime64(start)
    valid = (dates >= np.datetime64(start)) & (dates < np.datetime64(end))
    preds = []
    print(f"Validation {start}..{end}: {valid.sum()} cookies, {y[valid].sum()} bots")
    for name, model in models().items():
        model.fit(X_train.iloc[fit], y[fit])
        p = model.predict_proba(X_train.iloc[valid])[:, 1]
        preds.append(p)
        print(name, "P@R>=.7, PR-AUC, ROC-AUC", evaluate(y[valid], p))
    for w in [0.25, 0.5, 0.75]:
        p = w * preds[0] + (1-w) * preds[1]
        print("hist_weight", w, evaluate(y[valid], p))


Validation 2026-04-14..2026-04-17: 2367 cookies, 195 bots
hist P@R>=.7, PR-AUC, ROC-AUC (0.7366, np.float64(0.7712), np.float64(0.9282))
trees P@R>=.7, PR-AUC, ROC-AUC (0.5855, np.float64(0.7445), np.float64(0.9185))
hist_weight 0.25 (0.7044, np.float64(0.7656), np.float64(0.9253))
hist_weight 0.5 (0.7326, np.float64(0.7769), np.float64(0.9288))
hist_weight 0.75 (0.7085, np.float64(0.779), np.float64(0.9299))
Validation 2026-04-17..2026-04-20: 1951 cookies, 160 bots
hist P@R>=.7, PR-AUC, ROC-AUC (0.7943, np.float64(0.7903), np.float64(0.9388))
trees P@R>=.7, PR-AUC, ROC-AUC (0.7386, np.float64(0.764), np.float64(0.9209))
hist_weight 0.25 (0.7887, np.float64(0.7797), np.float64(0.9259))
hist_weight 0.5 (0.8175, np.float64(0.788), np.float64(0.9298))
hist_weight 0.75 (0.8028, np.float64(0.7907), np.float64(0.9337))


После удаления точных дубликатов равная смесь получила P@R≥0,7 **0,7326** на 14–16 апреля
и **0,8175** на 17–19 апреля. Это выборки по 2367 и 1951 куке, поэтому
показатели на скрытом test могут быть другими. Одна HistGradientBoosting
давала 0,7366 и 0,7943 соответственно. Удаление точных дублей улучшило
первую проверку смеси (с 0,7135 до 0,7326) и не изменило вторую с точностью
до 4 знаков. Равная смесь лучше одиночной модели в среднем по двум периодам.


## 5. Обучение на всём train и подготовка файла

Теперь метки всех обучающих дней можно использовать для финального обучения.
На test предсказываем только score. После записи ещё раз проверяем формат:
ровно две колонки, все test cookie по одному разу, нет пропусков и значения
лежат в `[0, 1]`. Отправлять нужно именно этот CSV, а не выбранный порог.


In [8]:
full = []
for name, model in models().items():
    model.fit(X_train, y)
    full.append(model.predict_proba(X_test)[:, 1])
# Округляем score до 10 знаков для воспроизводимости итогового CSV.
score = np.round(.5 * full[0] + .5 * full[1], 10)
submission = pd.DataFrame({"cookie_id": test.cookie_id, "score": score})
assert submission.columns.tolist() == ["cookie_id", "score"]
assert submission.cookie_id.is_unique and set(submission.cookie_id) == set(test.cookie_id)
assert submission.score.notna().all() and submission.score.between(0, 1).all()
submission.to_csv("submission.csv", index=False)
print("Готово:", len(submission), "строк; уникальных score:", submission.score.nunique())
display(submission.head())


Готово: 4909 строк; уникальных score: 4909


,cookie_id,score
0,ck_315fb710a0e371e7,0.008401
1,ck_a76ee3b3e3e522fd,0.179826
2,ck_94c9a4d382689e82,0.058849
3,ck_8eaf9509ad9462a0,0.008610
4,ck_9a88a5a989cb5bc6,0.011928


## Как интерпретировать результат

Модель выстраивает куки по степени похожести на размеченный трафик известных
сервисов. Её score удобен для ранжирования; без отдельной калибровки его не
следует трактовать как достоверную индивидуальную вероятность.

**Ограничения:** положительный класс содержит известные источники и может не
охватывать все разновидности ботов; метки «человеческого» трафика также даны
по источникам. Поэтому высокий показатель локальной проверки не гарантирует
такого же результата на других датах и новых типах автоматизации.
